In [2]:
from pathlib import Path
from collections import defaultdict
import re
import pandas as pd

# ====== CHANGE THIS ======
folder = Path(r"data/sept15_allcurrentlogfiles/FINALALLLOGFILES")
output_file = folder / "conformercheck.xlsx"
# =========================

pattern = re.compile(r"^(.*?)_crest_clustered-(\d+)\.log$")

# Store all Y values found for each XXX
xxx_values = defaultdict(set)

# Scan files
for file in folder.iterdir():
    if not file.is_file():
        continue

    match = pattern.match(file.name)

    if match:
        xxx = match.group(1)
        y = int(match.group(2))
        xxx_values[xxx].add(y)

# Build summary
rows = []

for xxx, values in sorted(xxx_values.items()):
    largest_y = max(values)

    # Expected sequence: 1, 2, 3, ..., largest_y
    expected = set(range(1, largest_y + 1))

    # Anything expected but not present is missing
    missing = sorted(expected - values)

    rows.append({
        "XXX": xxx,
        "Largest Y": largest_y,
        "Files Present": len(values),
        "Expected Files": largest_y,
        "Missing Y": ", ".join(map(str, missing)) if missing else "",
        "Complete?": "YES" if not missing else "NO"
    })

df = pd.DataFrame(rows)

# Write to Excel
df.to_excel(output_file, index=False)

print(f"Done. Results written to:\n{output_file}")

Done. Results written to:
data/sept15_allcurrentlogfiles/FINALALLLOGFILES/conformercheck.xlsx
